# Constant Acceleration MPC Testing

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_log_compiles", True)
jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# jax.config.update('jax_disable_jit', True)


import time
import functools
import jax.numpy as jnp
import numpy as np
import tqdm
import scipy.optimize as sci_opt
import scipy.linalg as sci_lin
import scipy.integrate as sci_int

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.quartic_cost as quartic_cost
import exp_mpc.stewart_min.mp_mpl as mp_mpl

In [ ]:
# general setup
n = 3200  # horizon
# n = 400  # horizon
# n = 200  # horizon
T = n / 200.0

acc_ref = jnp.array([1.0, 0.0, 0.0]) + const.gravity
omega_ref = jnp.array([0.0, 0.0, 0.0])

In [ ]:
# cost setup

weights = opt.ExpWeights(
    acc=jnp.array([1e1, 1e1, 1e0]),
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    low=const.leg_min,
    high=const.leg_max,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes = sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.max_yaw,
    high=const.max_yaw,
)

## compute fixed point

In [ ]:
def ell_x_base(x: jax.Array, u: jax.Array) -> jax.Array:
    state = opt.State(*x)
    control = opt.Control(*u)
    cost = jnp.array(0.0)
    cost += 0.5 * opt.head_xyz_acc_cost_single(acc_ref, state, control, weights.acc)
    cost += 0.5 * opt.omega_cost_single(omega_ref, state, weights.omega)

    # legs
    length, vel = opt.get_length_and_vel(state)
    cost += jnp.sum(weights.leg * jax.vmap(leg_cost)(length.flatten()))
    cost += jnp.sum(weights.leg_vel * jax.vmap(leg_vel_cost)(vel.flatten()))

    # joint angle
    angle = jnp.ravel(opt.get_joint_angles(state))
    cost += jnp.sum(weights.joint_angle * jax.vmap(joint_angle_cost)(angle))

    # yaw
    cost += jnp.squeeze(weights.yaw * yaw_cost(state.yaw))

    return cost

ell_x = jax.jit(ell_x_base)

dx_ell_x = jax.jit(jax.grad(ell_x, argnums=0))
du_ell_x = jax.jit(jax.grad(ell_x, argnums=1))

ddx_ell_x = jax.jit(jax.hessian(ell_x, argnums=0))

In [ ]:
res_x0 = sci_opt.minimize(
    fun=functools.partial(ell_x, u=jnp.zeros(6)),
    x0=jnp.zeros(12),
    method="L-BFGS-B",
    jac=functools.partial(dx_ell_x, u=jnp.zeros(6)),
    options={
        "maxiter": 2**8,
        "maxls": 2**8,
        # "maxiter": 16,
        # "maxls": 8,
    },
    tol=1e-16,
)
res_x0.x

## bvp setup

In [ ]:
# x0 = res_x0.x
x0 = jnp.zeros(12)
# x0 = jnp.array([
#     5.43502301e-02, 6.80128182e-04, 6.94407911e-03,
#     3.64224717e-06, -9.91089002e-02, -6.59085185e-04,
#     5.93086348e-03, 2.11808585e-05, 1.29081290e-03,
#     1.10529159e-05, -1.00385042e-02,  1.88575859e-04,
# ])  # 600
# x0 = jnp.array([
#     6.31694430e-02, 6.75132570e-04, 7.60997249e-03,
#     3.50138496e-06, -1.03179074e-01, -5.95061851e-04,
#     5.32385208e-03, -2.63263009e-05, 2.35695175e-04,
#     -9.61045180e-06,  3.61975505e-04, -2.01798928e-05,
# ])  # 800

In [ ]:
A = jnp.block([
    [jnp.zeros((6, 6)), jnp.identity(6)],
    [jnp.zeros((6, 6)), jnp.zeros((6, 6))],
])
B = jnp.block([
    [jnp.zeros((6, 6))], 
    [jnp.identity(6)],
])

In [ ]:
@jax.jit
def u_opt(x: jax.Array, p: jax.Array) -> jax.Array:
    W_ua = jnp.diag(jnp.square(weights.control[:3]))
    W_uw = jnp.diag(jnp.square(weights.control[3:]))
    W_a = jnp.diag(jnp.square(weights.acc))
    R = utils._get_R(*x[3:6])
    B_a = B[:, :3]
    B_w = B[:, 3:]
    scale_a = jnp.linalg.inv(W_ua + R @ W_a @ R.T)
    scale_w = jnp.linalg.inv(W_uw)
    u_a = T * B_a.T @ p + R @ W_a @ (acc_ref - R.T @ const.gravity)
    u_w = T * B_w.T @ p
    return jnp.concatenate([scale_a @ u_a, scale_w @ u_w])

def rot_error(x: jax.Array) -> jax.Array:
    W_a = jnp.diag(jnp.square(weights.acc))
    R = utils._get_R(*x[3: 6])
    return R @ W_a @ (acc_ref - R.T @ const.gravity)

In [ ]:
v0 = rot_error(res_x0.x)

def k_bc(x: jax.Array) -> jax.Array:
    v = x[6: 9]
    return 1.0 / T * jnp.sum(jnp.tanh(v * v0))

In [ ]:
@jax.jit
def pon_dyn(x: jax.Array, p: jax.Array, u: jax.Array) -> jax.Array:
    x_dot = A @ x + B @ u
    p_dot = -A.T @ p + 1 / T * jax.grad(ell_x, argnums=0)(x, u)
    return jnp.concatenate([x_dot, p_dot])

In [ ]:
@jax.jit
def fun_jax_single(y: jax.Array) -> jax.Array:
    x = y[:12]
    p = y[12:]
    u = u_opt(x, p)
    return pon_dyn(x, p, u)

def fun_np(_: np.ndarray, y: np.ndarray) -> np.ndarray:
    y = y.T  # wierd ordering
    res = np.array([fun_jax_single(yi) for yi in y])
    return np.transpose(res)  # wierd ordering

fun_jac_single = jax.jit(jax.jacrev(fun_jax_single))

def fun_jac_np(_: np.ndarray, y: np.ndarray) -> np.ndarray:
    y = y.T  # wierd ordering
    res = np.array([fun_jac_single(yi) for yi in y])
    return np.transpose(res, axes=(1, 2, 0))  # wierd ordering

k_bc_grad = jax.jit(jax.grad(k_bc))
k_bc_jac = jax.jit(jax.jacrev(k_bc_grad))

def bc_np(ya: np.ndarray, yb: np.ndarray) -> np.ndarray:
    e0 = ya[:12] - np.ravel(x0)
    # e1 = yb[12:] - 0  # unregulated
    e1 = yb[12:] + np.array(k_bc_grad(yb[:12]))
    return np.concatenate([e0, e1])

def bc_jac_np(_: np.ndarray, yb: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    ya_jac = sci_lin.block_diag(np.identity(12), np.zeros((12, 12)))
    # yb_jac = sci_lin.block_diag(np.zeros((12, 12)), np.identity(12))  # unregulated
    yb_jac = sci_lin.block_diag(np.array(k_bc_jac(yb[:12])), np.identity(12))
    return ya_jac, yb_jac

## bvp solve

In [ ]:
x = np.arange(n + 1, dtype=float) / 200.0
y = np.concatenate([np.array(x0), np.zeros(12)])
y = np.transpose(np.tile(y, reps=(x.size, 1)))

res = sci_int.solve_bvp(
    fun=fun_np,
    bc=bc_np,
    x=x,
    y=y,
    fun_jac=fun_jac_np,
    bc_jac=bc_jac_np,
    verbose=2,
    tol=1e-10,
    bc_tol=1e-10,
    max_nodes=2**16,
)

In [ ]:
t = jnp.arange(int(T / const.dt) + 1) * const.dt
res_x = np.transpose(res.sol(t))[:, :12]
res_p = np.transpose(res.sol(t))[:, 12:]
res_u = jax.vmap(u_opt)(res_x, res_p)

## misc

In [ ]:
for i, val in enumerate(1.0 / T * rot_error(x0)):
    print(f"{i:>2}: {val:>10.2e}")
print()
for i in range(6, 9):
    print(f"{i:>2}: {res_p[0, i]:>10.2e}")

1.0 / T * rot_error(x0) - res_p[0, 6:9]

In [ ]:
res_p[0]

In [ ]:
plt.close("all")
# for i in range(6):
#     plt.plot(res_u[:, i], label=f"{i}")
for i in range(12):
    plt.plot(res_p[:, i], label=f"{i}")
# for i in range(12):
#     plt.plot(res_x[:, i], label=f"{i}")
plt.legend()
plt.show()

In [ ]:
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    yaw_cost=yaw_cost,
)
opt_cost = opt._cost(
    weights=weights,
    cost_terms=cost_terms,
    acc_ref=jnp.tile(A=acc_ref, reps=(res_x.shape[0] - 1, 1)),
    omega_ref=jnp.tile(A=omega_ref, reps=(res_x.shape[0] - 1, 1)),
    state0=x0,
    vel_ref=None,
    control=opt.Control.from_flat(jnp.ravel(res_u[:-1])),
    # state=opt.State(*list(jnp.array(res_x.T))),
)
opt_cost

In [ ]:
def ell_u(u: jax.Array) -> jax.Array:
    return 0.5 * jnp.sum(jnp.square(u * weights.control))

def hamilton(x: jax.Array, u: jax.Array, p: jax.Array) -> jax.Array:
    return jnp.inner(p, A @ x + B @ u) - 1.0 / T * (ell_x(x, u) + ell_u(u))

h_u = jax.vmap(jax.grad(hamilton, argnums=1))(res_x, res_u, res_p)
# h_u

## ivp solve

In [ ]:
p0 = np.copy(res_p[0])
p0 += np.array([
    0.0, 0.0, 0.0,
    0.0, 0.0, 0.0,
    0.0, 1e-5, 0.0,
    0.0, 0.0, 0.0,
])
# p0 = np.concatenate([np.zeros(6), -1.0 / T * rot_error(x0), np.zeros(3)])
# y0 = np.concatenate([x0, p0])

# p0 = np.random.normal(size=12)
# p0 = np.zeros(12)
y0 = np.concatenate([np.zeros(12), p0])

def ivp_fun_np(_: np.ndarray, y: np.ndarray) -> np.ndarray:
    return np.array(fun_jax_single(y))

def ivp_jac_np(_: np.ndarray, y: np.ndarray) -> np.ndarray:
    return np.array(fun_jac_single(y))

res_ivp = sci_int.solve_ivp(
    fun=ivp_fun_np,
    t_span=[0.0, T],
    # t_eval=np.linspace(0.0, T, num=11453),
    # max_step=T / 11453,
    y0=y0,
    vectorized=False,
    dense_output=True,
    jac=ivp_jac_np,
    method="Radau",
    # method="DOP853",
)
res_ivp

In [ ]:
ts = jnp.arange(int(T / const.dt) + 1) * const.dt
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ivp_ps = res_ivp.sol(ts)[12:]
for i in range(12):
# for i in [6, 7, 8]:
# for i in [7]:
    ax.plot(ivp_ps[i], label=f"{i}")
ax.legend()
ax.set_ylim(-2, 0.5)
ax.set_xlim(-100, 3300)
plt.show()